# Chunking & Vector Databases
**Topics covered:** why we chunk · fixed-size, sentence, recursive and header-based chunking · overlap · chunk-size trade-offs
Chroma · collections · metadata filters · persistence · indexing your own notes

> **Where this fits:** RAG = *Retrieve* relevant text, then *Generate* an answer from it. Last notebook you built the search engine core (embeddings + cosine similarity). This notebook answers two questions: **what exactly do we embed?** (chunking) and **where do we store it so search stays fast?** (vector database).

In [1]:
# Install if not already present
# !pip install sentence-transformers chromadb

---
## PART 1: Why We Chunk

You cannot embed a whole 200-page manual as one vector, for three reasons:

1. **One vector, one blurry meaning.** An embedding compresses text into a single point. A page about 20 topics becomes an average of all of them, and matches none of them well.
2. **The LLM has limited context and you pay per token.** If retrieval returns the whole manual, you waste the context window on text that has nothing to do with the question.
3. **Precision.** The user wants the one paragraph that answers their question, not the whole book.

**Analogy: a textbook index.** The index doesn't point you to "the book", it points you to *page 142*. Chunking is how you cut the book into pages that are worth pointing to.

```
Long document --split--> [chunk 1] [chunk 2] [chunk 3] ... --embed--> one vector per chunk --store--> vector DB
                                                                              ^
                                    user question --embed--> search ----------+--> top-k chunks --> LLM
```

**The core tension:**

| Chunk too small | Chunk too big |
|---|---|
| Loses context ("it was fast" - what was?) | Meaning is blurred across many topics |
| Many tiny vectors, noisy results | Wastes LLM context, retrieves irrelevant text |

Good chunking is finding the middle. There is no universal best size, so we will measure it.

---
## PART 2: Chunking Strategies From Scratch

We build four chunkers by hand so you see exactly what libraries such as LangChain do for you later.

In [2]:
sample = '''Docker volumes keep data outside the container. When a container is removed, its writable layer is deleted. A named volume lives in Docker's own storage and survives the container.

Bind mounts are different. They map a real folder from your computer into the container. They are ideal for live-editing code during development.

Networks let containers find each other by name. On a user-defined network Docker runs a small DNS service. The API can then call the database using the service name instead of an IP address.

Docker Compose describes the whole system in one file. It creates the network and volumes for you and starts the services together.'''

print(len(sample), "characters,", len(sample.split()), "words")

653 characters, 108 words


### Strategy 1: Fixed-size chunks (the naive way)

Cut every N characters, no matter what. Simple, fast, and often wrong, because it cuts through the middle of sentences.

In [3]:
def fixed_size_chunks(text, size):
    return [text[i:i + size] for i in range(0, len(text), size)]

for i, c in enumerate(fixed_size_chunks(sample, 200)):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(repr(c))

--- chunk 0 (200 chars) ---
"Docker volumes keep data outside the container. When a container is removed, its writable layer is deleted. A named volume lives in Docker's own storage and survives the container.\n\nBind mounts are di"
--- chunk 1 (200 chars) ---
'fferent. They map a real folder from your computer into the container. They are ideal for live-editing code during development.\n\nNetworks let containers find each other by name. On a user-defined netw'
--- chunk 2 (200 chars) ---
'ork Docker runs a small DNS service. The API can then call the database using the service name instead of an IP address.\n\nDocker Compose describes the whole system in one file. It creates the network '
--- chunk 3 (53 chars) ---
'and volumes for you and starts the services together.'


Look at where the chunks start and end: words and sentences are sliced in half. A chunk that begins with `"ontainer is removed..."` is a worse search target than one that begins with a complete sentence.

### Strategy 2: Fixed-size with overlap

Let neighbouring chunks **share** some text, so an idea cut at a boundary still appears whole in at least one chunk.

**Analogy:** when subtitles split a long sentence across two screens, they repeat a word or two so you don't lose the thread.

```
chunk 1: [==================]
chunk 2:               [==================]
                       ^^^^^ overlap
```

In [4]:
def fixed_size_with_overlap(text, size, overlap):
    assert overlap < size, "overlap must be smaller than chunk size"
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

chunks = fixed_size_with_overlap(sample, size=200, overlap=50)
print(len(chunks), "chunks")
for a, b in zip(chunks, chunks[1:]):
    print("end of one  :", repr(a[-50:]))
    print("start of next:", repr(b[:50]))
    print()
    break   # show just the first boundary

5 chunks
end of one  : 'ge and survives the container.\n\nBind mounts are di'
start of next: 'ge and survives the container.\n\nBind mounts are di'



Overlap fixes lost context at boundaries but costs more chunks (and storage). A common starting point is **10-20% overlap**.

### Strategy 3: Sentence-based chunks

Split on sentence boundaries and pack whole sentences together until a size limit is reached. Chunks never start mid-sentence.

In [5]:
import re

def sentence_chunks(text, max_chars):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""
    for s in sentences:
        if current and len(current) + len(s) + 1 > max_chars:
            chunks.append(current)
            current = s
        else:
            current = f"{current} {s}".strip()
    if current:
        chunks.append(current)
    return chunks

for i, c in enumerate(sentence_chunks(sample, 250)):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)

--- chunk 0 (207 chars) ---
Docker volumes keep data outside the container. When a container is removed, its writable layer is deleted. A named volume lives in Docker's own storage and survives the container. Bind mounts are different.
--- chunk 1 (226 chars) ---
They map a real folder from your computer into the container. They are ideal for live-editing code during development. Networks let containers find each other by name. On a user-defined network Docker runs a small DNS service.
--- chunk 2 (215 chars) ---
The API can then call the database using the service name instead of an IP address. Docker Compose describes the whole system in one file. It creates the network and volumes for you and starts the services together.


### Strategy 4: Recursive splitting (the practical default)

Documents already have structure: sections, paragraphs, lines, sentences. Respect the **biggest natural boundary first** and only fall back to smaller ones when a piece is still too big:

```
try "\n\n" (paragraphs)  ->  piece still too big?  ->  try "\n" (lines)  ->  try ". " (sentences)  ->  try " " (words)
```

This is the idea behind LangChain's `RecursiveCharacterTextSplitter`, the most commonly used splitter in real RAG projects.

In [6]:
def recursive_split(text, max_chars, separators=("\n\n", "\n", ". ", " ")):
    text = text.strip()
    if len(text) <= max_chars:
        return [text] if text else []
    if not separators:                                  # last resort: hard cut
        return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]

    sep, rest = separators[0], separators[1:]
    pieces = text.split(sep)
    chunks, current = [], ""
    for piece in pieces:
        candidate = f"{current}{sep}{piece}" if current else piece
        if len(candidate) <= max_chars:
            current = candidate                         # keep packing
        else:
            if current:
                chunks.append(current)
            if len(piece) > max_chars:                  # piece itself too big: go finer
                chunks.extend(recursive_split(piece, max_chars, rest))
                current = ""
            else:
                current = piece
    if current:
        chunks.append(current)
    return chunks

for i, c in enumerate(recursive_split(sample, 250)):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)

--- chunk 0 (180 chars) ---
Docker volumes keep data outside the container. When a container is removed, its writable layer is deleted. A named volume lives in Docker's own storage and survives the container.
--- chunk 1 (145 chars) ---
Bind mounts are different. They map a real folder from your computer into the container. They are ideal for live-editing code during development.
--- chunk 2 (191 chars) ---
Networks let containers find each other by name. On a user-defined network Docker runs a small DNS service. The API can then call the database using the service name instead of an IP address.
--- chunk 3 (131 chars) ---
Docker Compose describes the whole system in one file. It creates the network and volumes for you and starts the services together.


Each chunk is now a coherent paragraph (or a group of them). That is far better to embed and far better to show an LLM.

### Strategy 5: Structure-aware chunking (headers)

If your documents are Markdown (like your notebooks!), the headers tell you the topic of each section. Split on headers and **keep the header path as metadata**. The chunk then knows *which section it came from*, and that is gold for retrieval and for citing sources.


**Idea:** Don't cut text blindly. Cut where the author already marked the boundaries (headings, paragraphs, code blocks, functions).

**Analogy:** Making index cards from a textbook. Fixed-size cutting slices mid-sentence. Structure-aware cutting makes one card per section, with the section title on top.

| Document | Natural cut points |
|---|---|
| Markdown / docs | `#`, `##`, `###` headers |
| HTML | `<h1>`, `<section>`, `<p>` |
| Code | functions, classes |
| Chat / email | one message per chunk |

**Why it's better**
- Each chunk is a complete idea, not a fragment.
- The header goes with the chunk, so "20 days" becomes "Leave Policy > Annual Leave: 20 days" and the embedding knows what it's about.
- The section name can be stored as metadata for filtering and citations.

**Watch out for**
- **Section too big:** split it again with the recursive splitter.
- **Section too tiny:** merge it with its neighbour.
- **`# comment` inside a code block** looks like a header. Track code fences to avoid this.
- **No structure** (raw text, scanned PDFs): fall back to recursive or fixed-size chunking.

**Standard recipe:** split by headers → split oversized sections recursively → attach `{section, source}` metadata to every chunk.

**Rule of thumb:** use structure first and fall back to size-based splitting only when there is none.


In [7]:
def markdown_sections(text):
    """Split markdown on headers. Returns [(header_path, body), ...]."""
    sections, path, body, in_code = [], [], [], False
    def flush():
        content = "\n".join(body).strip()
        if content:
            sections.append((" > ".join(path), content))
    for line in text.splitlines():
        if line.strip().startswith("```"):              # '#' inside a code block is a comment, not a header
            in_code = not in_code
        m = None if in_code else re.match(r'^(#{1,6})\s+(.*)', line)
        if m:
            flush()
            body = []
            level = len(m.group(1))
            path = path[:level - 1] + [m.group(2).strip()]
        else:
            body.append(line)
    flush()
    return sections

doc = '''# Docker Guide
Intro text about Docker.

## Volumes
Volumes keep data outside the container.

### Named volumes
Managed by Docker, best for databases.

## Networks
Containers find each other by name.'''

for path, body in markdown_sections(doc):
    print(f"[{path}]  ->  {body!r}")

[Docker Guide]  ->  'Intro text about Docker.'
[Docker Guide > Volumes]  ->  'Volumes keep data outside the container.'
[Docker Guide > Volumes > Named volumes]  ->  'Managed by Docker, best for databases.'
[Docker Guide > Networks]  ->  'Containers find each other by name.'


### Which strategy when?

| Strategy | Good for | Weakness |
|---|---|---|
| Fixed-size | Quick experiments, unstructured text dumps | Cuts mid-sentence |
| Fixed + overlap | Same, with less boundary loss | More chunks, still cuts mid-sentence |
| Sentence-based | Prose, articles, transcripts | Ignores paragraphs and sections |
| **Recursive** | **Default for most text** | Needs a sensible size choice |
| Header-based/structure-aware | Markdown, docs, wikis, code docs | Sections can be very long, so combine with recursive |

**In practice:** split by headers first, then run a recursive splitter on any section that is still too long.

---
## PART 3: Chunk Size Is a Trade-off, So Measure It

Let's test the "too small / too big" claim with real embeddings. We take your own Docker Part 2 notes as the corpus and ask the same question using three chunk sizes.

In [8]:
import json, pathlib
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')       # same model as the embeddings notebook

NOTEBOOKS = pathlib.Path("../../backend-and-systems/docker/notebooks")

def load_markdown(path):
    nb = json.loads(path.read_text())
    return "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "markdown")

part2 = load_markdown(NOTEBOOKS / "02_docker_volumes_networks_compose.ipynb")
print(f"Loaded {len(part2):,} characters from your Docker Part 2 notes")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 18,993 characters from your Docker Part 2 notes


In [9]:
query = "How do I keep database data after the container is deleted?"
q_vec = model.encode(query, normalize_embeddings=True)

for size in (150, 600, 3000):
    chunks = recursive_split(part2, size)
    vecs = model.encode(chunks, normalize_embeddings=True)
    scores = vecs @ q_vec                              # cosine similarity (vectors are normalized)
    best = int(np.argmax(scores))
    print("=" * 70)
    print(f"chunk size {size}: {len(chunks)} chunks, avg {np.mean([len(c) for c in chunks]):.0f} chars")
    print(f"best match score = {scores[best]:.3f}")
    print(chunks[best][:400])

chunk size 150: 194 chunks, avg 96 chars
best match score = 0.635
All your tables and rows are still there, because the data lived in the volume (the safe deposit box), not in the container.
chunk size 600: 40 chunks, avg 473 chars
best match score = 0.613
**Analogy: a hotel room.** You can rearrange the furniture, but when you check out, housekeeping resets the room. Anything you want to keep must go in a **safe deposit box** outside the room. That box is a **volume**.

Run the cell below. We write a file inside a container, remove the container, then start a fresh one.

The second run fails with *No such file*. This is why a database in a plain co
chunk size 3000: 7 chunks, avg 2711 chars
best match score = 0.550
You can run `docker rm -f db` and start a new Postgres container with the same `-v pgdata:...` flag. All your tables and rows are still there, because the data lived in the volume (the safe deposit box), not in the container.

```bash
docker rm -f db
docker run -d --name db 

**What the results show (read the text, not just the score):**
- **Small chunks (150):** the highest score, because a short chunk is *focused* on one idea. But it is a fragment: it says the data "lived in the volume" without showing how to do it. High score, thin answer.
- **Medium chunks (600):** a slightly lower score, but the chunk holds the whole idea (the problem, the analogy and the failing example), so an LLM can answer from it.
- **Large chunks (3000):** the lowest score. The answer is in there, but mixed with unrelated commands and topics, so it matches the question less strongly and sends the LLM a lot of noise.

**Lesson:** a similarity score tells you how *focused* a chunk is, not how *useful* it is. You judge chunk size by whether the retrieved text can actually answer the question, which is why RAG evaluation (a later notebook) matters.

**Rules of thumb to start from** (then measure on your own data):
- 200-800 characters, or roughly 256-512 tokens per chunk
- 10-20% overlap
- Never exceed your embedding model's max input (`model.max_seq_length`, which is 256 tokens for MiniLM; longer text is silently truncated)

In [10]:
print("max tokens this model embeds:", model.max_seq_length)

max tokens this model embeds: 256


> **Silent truncation trap:** if a chunk is longer than the model's limit, the model just ignores the rest. You pay to store it, but the extra text never influences the vector. Always keep chunks under the limit.

---
## PART 4: Why We Need a Vector Database

Your mini search engine from last notebook compared the query to **every** vector:

```python
scores = vecs @ q_vec     # brute force: compare against ALL chunks
```

| Chunks | Brute-force comparisons per query |
|---|---|
| 1,000 | 1,000 (instant) |
| 1,000,000 | 1,000,000 (slow) |
| 100,000,000 | 100,000,000 (too slow) |

A **vector database** gives you:

1. **Approximate nearest neighbour (ANN) indexes** such as HNSW. Instead of checking everything, they navigate a "map" of the vectors and check only a small fraction. Slightly approximate, massively faster.
2. **Storage of text and metadata** beside each vector, so a hit returns the original text, not just a number.
3. **Metadata filtering** ("only chunks from the Docker notes").
4. **Persistence**, so you don't re-embed everything each time you restart.
5. **Updates and deletes** for changing documents.

**Analogy:** brute force is reading every book in the library to find one fact. An ANN index is the library's catalogue and floor map: you walk straight to the right shelf.

We will use **Chroma**: free, runs locally with no server, and has the same concepts as the bigger systems (Pinecone, Qdrant, Weaviate, pgvector).

| Chroma word | Plain meaning | SQL equivalent |
|---|---|---|
| **Client** | Connection to the database | connection |
| **Collection** | A group of related chunks | table |
| **Document** | The chunk text | row's text column |
| **Embedding** | The chunk's vector | row's vector column |
| **Metadata** | Extra fields (source, section...) | other columns |
| **ID** | Unique key | primary key |

HNSW stands for Hierarchical Navigable Small World. It's the algorithm most vector databases, including Chroma, use to find the nearest vectors quickly.

## The problem it solves

With 1 million chunks, a query has to be compared against every vector to find the closest ones. That's exact search, and it's slow (O(n)). Approximate nearest neighbour (ANN) search gives up a tiny bit of accuracy for a huge speedup. HNSW is the most popular ANN method.

## Analogy: finding a house in a new country

You don't check every house. You:

1. Pick the country (a few big jumps).
2. Pick the city.
3. Pick the street.
4. Walk house to house on that street.

HNSW builds the same kind of layered map over your vectors.

## How it works

- **Graph:** each vector is a node, linked to some of its close neighbours.
- **Layers:** the top layer has only a few nodes with long-range links (the "country" level). Each layer below has more nodes and shorter links. The bottom layer has all of them.
- **Search:**
  1. Start at the top layer and hop toward the node closest to your query.
  2. When you can't get closer, drop down a layer.
  3. Repeat until the bottom layer, where you collect the final nearest neighbours.

"Small world" refers to the idea that any node can reach any other in just a few hops, like "six degrees of separation".

---
## PART 5: Chroma Basics

In [11]:
import chromadb

client = chromadb.Client()                              # in-memory: gone when the kernel stops

collection = client.get_or_create_collection(
    name="demo",
    metadata={"hnsw:space": "cosine"},                  # use cosine distance for the index
)
print("collection:", collection.name, "| items:", collection.count())

collection: demo | items: 0


We pass our **own embeddings** (from the same model for documents and queries). That keeps full control and makes the point that the database doesn't understand text. It only stores and compares vectors.

In [12]:
docs = [
    "Named volumes keep database data after the container is deleted.",
    "Bind mounts map a folder from your computer into the container.",
    "Containers on the same network can reach each other by service name.",
    "A healthcheck tells Docker whether the service is actually ready.",
    "Python decorators wrap a function to add behaviour.",
]
ids   = [f"doc{i}" for i in range(len(docs))]
metas = [{"topic": "docker"}] * 4 + [{"topic": "python"}]

collection.add(
    ids=ids,
    documents=docs,
    embeddings=model.encode(docs, normalize_embeddings=True).tolist(),
    metadatas=metas,
)
print("items:", collection.count())

items: 5


In [13]:
def search(collection, text, k=3, where=None):
    result = collection.query(
        query_embeddings=[model.encode(text, normalize_embeddings=True).tolist()],
        n_results=k,
        where=where,
    )
    for doc, dist, meta in zip(result["documents"][0], result["distances"][0], result["metadatas"][0]):
        print(f"similarity={1 - dist:.3f}  {meta}  {doc}")

search(collection, "How do I stop losing my database rows?")

similarity=0.382  {'topic': 'docker'}  Named volumes keep database data after the container is deleted.
similarity=0.043  {'topic': 'docker'}  A healthcheck tells Docker whether the service is actually ready.
similarity=0.032  {'topic': 'docker'}  Bind mounts map a folder from your computer into the container.


Chroma returns **distance**, not similarity. With cosine distance, `similarity = 1 - distance`, and smaller distance means a closer match.

### Metadata filtering

Search only inside a subset. This is how you build "search only this user's documents" or "only the 2024 reports".

## Why Chroma returns distance

Chroma doesn't implement its own vector search. It uses an HNSW index (built on the hnswlib library) to do that. HNSW is a graph-navigation algorithm, and navigating means "move to whichever neighbour is closer to the query". That needs one rule across all metrics: smaller number = closer.

So hnswlib exposes every metric as a distance:

| Space | Distance formula | What it means |
|---|---|---|
| `l2` (Chroma's default) | squared Euclidean distance | straight-line gap |
| `cosine` | 1 - cos_sim | angle gap |
| `ip` (inner product) | 1 - dot_product | dot-product gap |

With one rule, the same search code works for any metric. Chroma passes the numbers through unchanged, and converting to a "similarity" is left to you.

## How it works internally

When you do `collection.add(documents=..., metadatas=..., ids=...)`, three things happen:

1. **Embed:** the text becomes a vector (by the embedding function you pass or Chroma's default).
2. **Store the vector** in the HNSW index, a layered graph on disk.
3. **Store the text, ids, and metadata** in a normal database (SQLite for a local persistent client).

So there are two stores that work together:

```
SQLite            → id, document text, metadata   (filtering, lookup)
HNSW index        → id, vector, graph links       (nearest-neighbour search)
```

When you run `collection.query(query_texts=["leave policy"], n_results=3)`:

1. **Embed** the query into a vector.
2. **Apply the metadata filter**, if you gave a `where=`, to decide which ids are allowed.
3. **Search HNSW:** enter at the top layer, hop toward the query, drop a layer, and repeat until the bottom layer. A priority queue holds the best candidates found so far, keyed by distance. Results come out smallest distance first.
4. **Look up** each winning id in SQLite to get its text and metadata.
5. **Return** ids, documents, metadatas, and distances, all in the same order.

## Why results are lists of lists

`results["distances"][0]` is the answer for the first query. If you pass several `query_texts`, you get one inner list per query, which is why we index `[0]`.  

### Two things to remember

**1. The search is approximate.**
Like finding the cheapest petrol station by driving toward the cheap-looking area instead of visiting every station: much faster, but a side-street winner can occasionally be missed. For RAG (top 3 to 5 chunks) this rarely matters, and on a small collection it is effectively exact.

**2. Distance only makes sense for the space it was created with.**
Like the number "5" meaning nothing without a unit (km, miles, minutes). In `cosine` space, distance 0.3 means similarity 0.7. In `l2` space, `1 - distance` is **not** a valid similarity, so a threshold like `min_similarity=0.25` would keep or reject the wrong chunks. With normalized embeddings and cosine in mind, set `cosine` explicitly at creation instead of using the default `l2`.


In [14]:
print("Only python chunks:")
search(collection, "How do I stop losing my database rows?", where={"topic": "python"})

Only python chunks:
similarity=-0.083  {'topic': 'python'}  Python decorators wrap a function to add behaviour.


Even though a python chunk is a poor match, the filter forces the search to stay within it. Filters are applied *with* the vector search, not after, so you still get k results from the subset.

Filter operators: `{"topic": "docker"}`, `{"year": {"$gte": 2024}}`, `{"$and": [ ... ]}`, `{"topic": {"$in": ["docker", "python"]}}`.

### Update and delete

In [15]:
collection.update(ids=["doc4"], documents=["Python generators yield values lazily."],
                  embeddings=model.encode(["Python generators yield values lazily."], normalize_embeddings=True).tolist())
collection.delete(ids=["doc3"])
print("items now:", collection.count())
print(collection.get(ids=["doc4"])["documents"])

items now: 4
['Python generators yield values lazily.']


**Important:** IDs must be unique and stable. If you re-add the same ID, you overwrite or duplicate depending on the method, so derive IDs from the source (for example `"docker_part2:12"`), not from a random counter.

### Persistence

`chromadb.Client()` lives in memory. `PersistentClient(path=...)` writes to disk, so the index survives restarts. Embedding is the slow, costly part, so you never want to redo it needlessly.

In [16]:
persistent = chromadb.PersistentClient(path="./chroma_store")
print("collections on disk:", [c.name for c in persistent.list_collections()])

collections on disk: []


---
## PART 6: Index Your Own Notes (a Mini Real Pipeline)

Now the full ingestion pipeline on real documents: your two Docker notebooks.

```
.ipynb files -> markdown text -> split by headers -> recursive split of long sections
             -> attach metadata (source, section) -> embed -> store in Chroma
```

In [17]:
def build_chunks(path, max_chars=600):
    text = load_markdown(path)
    out = []
    for section_path, body in markdown_sections(text):
        for j, piece in enumerate(recursive_split(body, max_chars)):
            out.append({
                "id": f"{path.stem}:{len(out)}",
                "text": piece,
                "meta": {"source": path.name, "section": section_path or "(top)"},
            })
    return out

all_chunks = []
for nb_path in sorted(NOTEBOOKS.glob("*.ipynb")):
    chunks = build_chunks(nb_path)
    print(f"{nb_path.name}: {len(chunks)} chunks")
    all_chunks += chunks
print("total chunks:", len(all_chunks))

01_docker_fundamentals.ipynb: 31 chunks
02_docker_volumes_networks_compose.ipynb: 44 chunks
total chunks: 75


In [18]:
notes = persistent.get_or_create_collection("docker_notes", metadata={"hnsw:space": "cosine"})

# Changed your chunking? Re-index with: persistent.delete_collection("docker_notes")
if notes.count() == 0:                                  # index only once
    texts = [c["text"] for c in all_chunks]
    notes.add(
        ids=[c["id"] for c in all_chunks],
        documents=texts,
        embeddings=model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist(),
        metadatas=[c["meta"] for c in all_chunks],
    )
print("indexed:", notes.count())

indexed: 75


In [19]:
for question in [
    "Why does my container lose its data?",
    "What does 0.0.0.0 mean?",
    "How do I run several copies of my API?",
]:
    print("\nQ:", question)
    search(notes, question, k=2)


Q: Why does my container lose its data?
similarity=0.596  {'section': 'Docker: Part 2 > 1. The Problem: Containers Forget Everything', 'source': '02_docker_volumes_networks_compose.ipynb'}  A container's filesystem is a thin writable layer on top of the read-only image. **Delete the container and that layer is gone.**

**Analogy: a hotel room.** You can rearrange the furniture, but when you check out, housekeeping resets the room. Anything you want to keep must go in a **safe deposit box** outside the room. That box is a **volume**.

Run the cell below. We write a file inside a container, remove the container, then start a fresh one.

The second run fails with *No such file*. This is why a database in a plain container loses all its rows when the container is replaced.

---
similarity=0.558  {'source': '02_docker_volumes_networks_compose.ipynb', 'section': 'Docker Volumes: Persisting Postgres Data > 4. Docker Compose: One File, Whole System > Prove the volume works'}  `docker compose 

In [20]:
# Restrict to one notebook using metadata
search(notes, "How do containers talk to each other?", k=2,
       where={"source": "02_docker_volumes_networks_compose.ipynb"})

similarity=0.471  {'section': 'Docker: Part 2 > Volumes, Networks & Docker Compose', 'source': '02_docker_volumes_networks_compose.ipynb'}  **Topics covered:**
1. The Problem: Containers Forget Everything
2. Volumes (named volumes vs bind mounts)
3. Networks: How Containers Find Each Other
4. Docker Compose: One File, Whole System
5. Health Checks & Startup Order
6. Scaling an Image into Many Containers
7. Hardening: `.dockerignore` & Non-Root User
8. Cleanup & Cheat Sheet

---
> **Context:** In Part 1 you ran one container. Real systems are an API + a database + a cache. Part 2 is how those pieces keep their data and talk to each other. This is the system design diagram, made real.
similarity=0.414  {'section': 'Part 2 Summary: Volumes, Networks & Compose > 2. Networks: containers finding each other', 'source': '02_docker_volumes_networks_compose.ipynb'}  - `localhost` inside a container means that container itself, not its neighbour.
- IP addresses change on every restart, so never r

Each result carries its **source and section**. In a real RAG app, you show these as citations ("answer found in *Docker Part 2 > Networks*"), which makes answers trustworthy and checkable.

You have just built the retrieval half of RAG. In the next notebook the retrieved chunks get handed to an LLM to write the answer.

---
## PART 7: Common Mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Different embedding models for indexing and querying | Random, meaningless results | Always use the *same* model for both |
| Chunks over the model's token limit | Answers hidden in the truncated tail are never found | Check `max_seq_length`, keep chunks under it |
| No overlap, hard cuts | Answers split across two chunks and neither matches | Recursive splitting, 10-20% overlap |
| Forgetting the metric | Wrong ordering | Set cosine on the collection, normalize vectors |
| Re-embedding on every run | Slow, wasteful | `PersistentClient` and index once |
| Unstable IDs | Duplicates after re-indexing | IDs derived from source and position |
| No metadata | Cannot filter or cite | Store source, section, page, date |
| Treating top-1 as truth | Confidently wrong answers | Retrieve top-k, check scores, evaluate |

---
## Key Takeaways

| Concept | Remember |
|---|---|
| **Why chunk** | One vector per idea. Too big blurs meaning, too small loses context |
| **Recursive splitting** | Respect paragraphs, then sentences, then words. The default choice |
| **Overlap** | Protects ideas cut at boundaries, 10-20% is typical |
| **Header-based** | Keep the section path as metadata for filtering and citations |
| **Chunk size** | 200-800 chars to start, always measure on your own data |
| **Vector DB** | ANN index + text + metadata + persistence, not just a list of vectors |
| **Chroma** | client -> collection -> add / query / get / update / delete |
| **Distance vs similarity** | `similarity = 1 - cosine distance` |
| **Same model** | Index and query with the same embedding model |